In [1]:
"""
===================================================================================
주요 라이브러리:
- pandas: 데이터프레임 처리
- numpy: 수치 연산
- json: JSON 파일 읽기
- requests: 카카오 지오코딩 API 호출 (필요 시)
===================================================================================
"""


import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import time
import glob
from scipy.spatial import cKDTree
from math import radians, sin, cos, asin, sqrt
from pathlib import Path


plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 데이터 로드 및 확인

In [2]:
"""
===================================================================================
1. 파일 경로 설정 및 라이브러리 임포트
===================================================================================
목적: 애견온도(Pet Temperature) 계산에 필요한 라이브러리와 데이터 파일 경로를 설정합니다.
===================================================================================
"""


# 프로젝트 루트 경로 설정
BASE_DIR = Path(r"c:\dev\study\team_project\Final Project\SKN18-FINAL-1TEAM")

# 데이터 경로 변수 정의
PET_DIR = BASE_DIR / "data" / "GraphDB_data" / "pet"
PET_PLAYGROUND_CSV = PET_DIR / "반려동물 놀이터.csv" # <--- 이 부분이 정의되어야 합니다.

STORE_DIR = BASE_DIR / "data" / "GraphDB_data" / "store_data"
PET_DATA_CSV = STORE_DIR / "소상공인시장진흥공단_상가(상권)정보_서울_cleaned.csv"

PARK_DIR = BASE_DIR / "data" / "GraphDB_data" / "park"
LAND_DIR = BASE_DIR / "data" / "GraphDB_data" / "land"
LISTING_JSON = LAND_DIR / "00_통합_원투룸.json"

print("✅ 환경 설정 및 경로 정의 완료")

✅ 환경 설정 및 경로 정의 완료


In [3]:
"""
===================================================================================
2. 데이터 로드 및 정교한 필터링 (업종 및 지오코딩 강화)
===================================================================================
목적: 반려동물 전문 시설을 선별하고, 좌표가 없는 놀이터 데이터의 위치를 추출합니다.
===================================================================================
"""
import os

# --- 지오코딩 함수 정의 ---
def get_coords_from_kakao(address):
    """카카오 API를 이용해 주소를 좌표로 변환"""
    kakao_key = os.environ.get("KAKAO_API_KEY") # .env에 설정된 키 사용
    if not address or not kakao_key:
        return None, None
        
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {kakao_key}"}
    params = {"query": address}
    
    try:
        response = requests.get(url, headers=headers, params=params, timeout=5)
        if response.status_code == 200:
            data = response.json()
            if data['documents']:
                return float(data['documents'][0]['y']), float(data['documents'][0]['x'])
    except:
        pass
    return None, None

# 1. 반려동물 놀이터 로드 및 초기화
print("⌛ 반려동물 놀이터 로딩 및 지오코딩 중...")
try:
    df_playground = pd.read_csv(PET_PLAYGROUND_CSV, encoding='utf-8')
except:
    df_playground = pd.read_csv(PET_PLAYGROUND_CSV, encoding='cp949')

# 좌표가 없으면 주소를 기반으로 추출
if '위치' in df_playground.columns:
    print("📍 놀이터 좌표 추출 중 (API 호출)...")
    coords = df_playground['위치'].apply(get_coords_from_kakao)
    df_playground['위도'] = [c[0] for c in coords]
    df_playground['경도'] = [c[1] for c in coords]
    
# 좌표가 생기거나 기존에 있던 데이터 중 유효한 것만 남김
df_playground = df_playground.dropna(subset=['위도', '경도'])
print(f"✅ 반려동물 놀이터: {len(df_playground)}개 (좌표 확보 완료)")

# 2. 상가 데이터 로드
print("⌛ 상가 데이터 로딩 중...")
df_stores = pd.read_csv(PET_DATA_CSV, low_memory=False)

# 필터링 키워드
pet_keywords = ['애견', '반려', '펫', '강아지', '고양이', '도그', '캣', '독', '멍멍', '야옹', '댕댕', '그루밍', '퍼피', '동반']

def contains_pet_kw(text):
    if pd.isna(text): return False
    text_clean = str(text).replace(" ", "").lower()
    return any(kw in text_clean for kw in pet_keywords)

# 분류별 추출 (강화된 필터링)
df_hospital = df_stores[df_stores['상권업종소분류명'] == '동물병원'].copy()
df_supplies = df_stores[df_stores['상권업종소분류명'] == '애완동물/애완용품 소매업'].copy()

df_grooming = df_stores[
    ((df_stores['상권업종소분류명'].apply(contains_pet_kw)) | (df_stores['상호명'].apply(contains_pet_kw))) & 
    (df_stores['상권업종소분류명'].str.contains('미용|애견|반려|동물', na=False)) &
    (~df_stores['상권업종소분류명'].str.contains('피부|네일|메이크업', na=False))
].copy()

df_cafe = df_stores[
    (df_stores['상권업종소분류명'].str.contains('카페|커피숍|디저트', na=False)) & 
    (df_stores['상호명'].apply(contains_pet_kw)) &
    (~df_stores['상권업종소분류명'].str.contains('독서실|스터디|실버|만화', na=False))
].copy()

print(f"✅ 동물병원: {len(df_hospital)}개")
print(f"✅ 애완용품점: {len(df_supplies)}개")
print(f"✅ 애견미용실: {len(df_grooming)}개")
print(f"✅ 애견동반카페: {len(df_cafe)}개")

# 3. 공원 데이터 통합 (116㎡ 이상)
print("⌛ 도시공원 통합 중...")
park_files = glob.glob(str(PARK_DIR / "*.csv"))
df_list = []
for file in park_files:
    try: temp_df = pd.read_csv(file, encoding='cp949')
    except: temp_df = pd.read_csv(file, encoding='utf-8')
    if '공원면적' in temp_df.columns:
        temp_df['공원면적_num'] = pd.to_numeric(temp_df['공원면적'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
        temp_df = temp_df[temp_df['공원면적_num'] >= 116]
        df_list.append(temp_df)
df_parks = pd.concat(df_list, ignore_index=True)
print(f"✅ 도시공원 (116㎡+): {len(df_parks)}개")

print("\n" + "=" * 60)
print("🎉 모든 데이터 준비 완료 (놀이터 좌표 포함)")
print("=" * 60)

⌛ 반려동물 놀이터 로딩 및 지오코딩 중...
📍 놀이터 좌표 추출 중 (API 호출)...
✅ 반려동물 놀이터: 15개 (좌표 확보 완료)
⌛ 상가 데이터 로딩 중...
✅ 동물병원: 873개
✅ 애완용품점: 1528개
✅ 애견미용실: 667개
✅ 애견동반카페: 111개
⌛ 도시공원 통합 중...
✅ 도시공원 (116㎡+): 1707개

🎉 모든 데이터 준비 완료 (놀이터 좌표 포함)


In [4]:
# 각 데이터프레임의 상위 5개 행 출력
print("🔍 [1. 반려동물 놀이터 샘플]")
display(df_playground.head(5))

print("\n🔍 [2. 동물병원 샘플]")
display(df_hospital[['상호명', '상권업종소분류명', '경도', '위도']].head(5))

print("\n🔍 [3. 애견용품점 샘플]")
display(df_supplies[['상호명', '상권업종소분류명', '경도', '위도']].head(5))

print("\n🔍 [4. 애견미용실 샘플]")
display(df_grooming[['상호명', '상권업종소분류명', '경도', '위도']].head(5))

print("\n🔍 [5. 애견동반카페 샘플]")
display(df_cafe[['상호명', '상권업종소분류명', '경도', '위도']].head(5))

print("\n🔍 [6. 도시공원 샘플]")
display(df_parks[['공원명', '공원면적', '경도', '위도']].head(5))

🔍 [1. 반려동물 놀이터 샘플]


,공원명,규모(㎡),위치,운영시간,전화번호,위도,경도
0,북서울꿈의숲,815,서울 강북구 월계로 173 (번동),10:00 ~ 19:00 *하절기 20시까지,02-901-6463,37.620254,127.044325
1,어린이대공원,747,광진구 능동로 216(구의문주차장 옆),24시간 상시개방['25.7.23.(수) 휴장],02-2124-2835,37.552012,127.078770
2,구로구 반려견 놀이터,700,구로구 신도림동 285-34일대(안양천 오금교남단),24시간 상시개방,02-860-2614,37.507121,126.874489
3,구로구 반려견 놀이터,2600,서울 구로구 구로2동 621-8일대(안양천 고척교 하단),24시간 상시개방,02-860-2614,37.499939,126.870524
4,금천구 반려견 놀이터,757,서울 금천구 시흥동 784-21(안양천변 유휴공간),24시간 상시개방,02-2627-2592,37.451646,126.895211



🔍 [2. 동물병원 샘플]


,상호명,상권업종소분류명,경도,위도
250,피움동물병원,동물병원,126.933155,37.649806
514,유림동물병원,동물병원,127.069551,37.622388
758,안정택동물병원,동물병원,126.838832,37.485914
1659,신내동물병원,동물병원,127.085738,37.602745
1702,날으는동물병원,동물병원,126.952166,37.548932



🔍 [3. 애견용품점 샘플]


...

,상호명,상권업종소분류명,경도,위도
1796,빅펫,애완동물/애완용품 소매업,127.058235,37.506048
2628,미주애견샵,애완동물/애완용품 소매업,127.037394,37.558351
3281,리퓨어펫피언,애완동물/애완용품 소매업,126.886201,37.468283
3408,그루밍하와이,애완동물/애완용품 소매업,126.947623,37.508078
3551,고양이정원,애완동물/애완용품 소매업,126.803172,37.575708



🔍 [5. 애견동반카페 샘플]


,상호명,상권업종소분류명,경도,위도
111,언더독커피,카페,126.917524,37.561797
747,프라빈커피&브레드독바위역점,카페,126.933251,37.618254
10614,달리는커피독산,카페,126.897104,37.464076
10923,스탠퍼독,카페,127.029698,37.644217
19229,282독,카페,127.025507,37.516683



🔍 [6. 도시공원 샘플]


,공원명,공원면적,경도,위도
0,신사근린공원,6800.0,127.021265,37.526708
1,대청근린공원,14089.1,127.083374,37.492416
2,늘푸른근린공원,11410.3,127.077339,37.489511
3,개포서근린공원,11219.2,127.063558,37.488594
4,개포동근린공원,9705.6,127.068291,37.489706


In [5]:
"""
===================================================================================
3. 매물 데이터 로드
===================================================================================
목적: 분석할 부동산 매물(JSON) 데이터를 불러옵니다.
===================================================================================
"""
print(f"⌛ 매물 데이터 로딩 중: {LISTING_JSON.name}")
with open(LISTING_JSON, 'r', encoding='utf-8') as f:
    land_data = json.load(f)

# 리스트 형태로 변환 (JSON 구조에 따라 다를 수 있음)
if isinstance(land_data, dict):
    listings = list(land_data.values()) if not isinstance(land_data, list) else land_data
else:
    listings = land_data

print(f"✅ 매물 데이터 로드 완료: {len(listings)}개")

⌛ 매물 데이터 로딩 중: 00_통합_원투룸.json
✅ 매물 데이터 로드 완료: 4683개


# 거리 분석

각 매물에서 가장 가까운 시설까지의 직선 거리를 계산합니다.

In [6]:
from scipy.spatial import cKDTree

# --- 컬럼명 표준화 함수 ---
def standardize_coords(df):
    """다양한 형태의 위경도 컬럼명을 lat, lon으로 통일"""
    cols = df.columns.tolist()
    rename_dict = {}
    
    # 위도 후보군
    for lat_name in ['위도', 'lat', 'latitude', '좌표_정보.위도', 'y']:
        if lat_name in cols:
            rename_dict[lat_name] = 'lat'
            break
            
    # 경도 후보군
    for lon_name in ['경도', 'lon', 'longitude', '좌표_정보.경도', 'x']:
        if lon_name in cols:
            rename_dict[lon_name] = 'lon'
            break
            
    df = df.rename(columns=rename_dict)
    
    # 숫자형 변환 및 결측치 제거
    if 'lat' in df.columns and 'lon' in df.columns:
        df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
        df['lon'] = pd.to_numeric(df['lon'], errors='coerce')
        return df.dropna(subset=['lat', 'lon'])
    return df

# 1. 매물 데이터 표준화
properties_df = pd.json_normalize(listings)
properties_df = standardize_coords(properties_df)

# 2. 시설 데이터 딕셔너리 및 표준화
raw_facilities = {
    'playground': df_playground,
    'hospital': df_hospital,
    'grooming': df_grooming,
    'cafe': df_cafe,
    'supplies': df_supplies,
    'park': df_parks
}

facilities = {}
for name, df in raw_facilities.items():
    standard_df = standardize_coords(df)
    if 'lat' in standard_df.columns and 'lon' in standard_df.columns:
        facilities[name] = standard_df
        print(f"✅ {name}: {len(standard_df)}개 로드 완료")
    else:
        print(f"❌ {name}: 위경도 컬럼을 찾을 수 없습니다. (현재 컬럼: {df.columns.tolist()})")

print(f"\n🚀 매물 데이터: {len(properties_df)}개 준비 완료")

✅ playground: 15개 로드 완료
✅ hospital: 873개 로드 완료
✅ grooming: 667개 로드 완료
✅ cafe: 111개 로드 완료
✅ supplies: 1528개 로드 완료
✅ park: 1707개 로드 완료

🚀 매물 데이터: 4683개 준비 완료


In [7]:
def get_nearest_distance(sources_df, targets_df):
    if targets_df.empty:
        return np.full(len(sources_df), np.inf)
    
    def to_xy(df):
        # 만약 여기서 에러가 난다면 데이터에 lat/lon 컬럼이 없는 것임
        x = df['lon'].values * 88000
        y = df['lat'].values * 111000
        return np.column_stack([x, y])
    
    source_xy = to_xy(sources_df)
    target_xy = to_xy(targets_df)
    
    tree = cKDTree(target_xy)
    dists, _ = tree.query(source_xy, k=1)
    return dists

results = properties_df.copy()

for fac_name, fac_df in facilities.items():
    print(f"⌛ {fac_name} 최단 거리 계산 중...")
    results[f'dist_{fac_name}'] = get_nearest_distance(results, fac_df)

# --- 반려동물 점수 산출 ---
def calculate_pet_score(row):
    s_pg = max(0, (1000 - row.get('dist_playground', 1000)) / 1000) * 30
    s_h = max(0, (1000 - row.get('dist_hospital', 1000)) / 1000) * 25
    s_gr = max(0, (700 - row.get('dist_grooming', 700)) / 700) * 15
    s_cf = max(0, (700 - row.get('dist_cafe', 700)) / 700) * 10
    s_sp = max(0, (700 - row.get('dist_supplies', 700)) / 700) * 10
    s_pk = max(0, (800 - row.get('dist_park', 800)) / 800) * 10
    return s_pg + s_h + s_gr + s_cf + s_sp + s_pk

results['raw_score'] = results.apply(calculate_pet_score, axis=1)
results['pet_temp'] = (30.0 + (13.0 * (results['raw_score'] / 100.0))).round(1)

print("\n🎉 모든 계산이 완료되었습니다!")
# address 컬럼명은 주소_정보.전체주소 등이 반영되도록 위에서 rename하거나 아래처럼 확인
display(results.sort_values(by='pet_temp', ascending=False).head(10))

⌛ playground 최단 거리 계산 중...
⌛ hospital 최단 거리 계산 중...
⌛ grooming 최단 거리 계산 중...
⌛ cafe 최단 거리 계산 중...
⌛ supplies 최단 거리 계산 중...
⌛ park 최단 거리 계산 중...

🎉 모든 계산이 완료되었습니다!


,매물번호,lat,lon,dist_playground,dist_hospital,dist_grooming,dist_cafe,dist_supplies,dist_park,raw_score,pet_temp
3473,18390934,37.498357,127.099102,248.388762,349.885190,30.678906,2328.623454,30.678906,312.902272,68.794254,38.9
1497,18431507,37.566276,127.056359,629.836506,60.530665,178.366554,213.690437,178.366554,121.466353,68.650354,38.9
3471,18436863,37.498150,127.098444,264.351531,347.788781,40.718028,2379.637916,40.718028,364.616450,67.362814,38.8
1480,18355064,37.552588,127.067959,169.702078,363.813044,184.776465,1583.704146,184.776465,133.549885,67.545079,38.8
1474,18293229,37.552247,127.067994,172.048880,338.640766,199.004611,1545.652849,199.004611,125.673140,67.694293,38.8
1477,18354573,37.551063,127.070185,392.649573,105.395567,237.398709,1411.995448,195.147044,149.652157,65.840042,38.6
2,18448077,37.474999,127.041886,483.124018,32.163128,292.207434,1567.256890,34.662820,127.192491,66.355524,38.6
5,18443369,37.474999,127.041886,483.124018,32.163128,292.207434,1567.256890,34.662820,127.192491,66.355524,38.6
1475,18451563,37.552688,127.071019,438.795091,240.661745,111.964212,1597.384016,69.998684,148.080055,65.569389,38.5
1471,18457380,37.552688,127.071019,438.795091,240.661745,111.964212,1597.384016,69.998684,148.080055,65.569389,38.5
